In [ ]:
import pandas as pd
from optbinning import OptimalBinning

train = pd.read_csv("../data/processed/train.csv")
validation = pd.read_csv("../data/processed/validation.csv")
test = pd.read_csv("../data/processed/test.csv")

In [ ]:
selected_variables = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "MonthlyIncome",
    "DebtRatio",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberRealEstateLoansOrLines",
    "NumberOfDependents",
    "NumberOfTimes90DaysLate"
]

In [ ]:
x = train["age"]
y = train["SeriousDlqin2yrs"]

optb_age = OptimalBinning(
    name="age",
    dtype="numerical"
)

optb_age.fit(x, y)

In [ ]:
train["age_woe"] = optb_age.transform(
    train["age"],
    metric="woe"
)

In [ ]:
train[
    ["age", "age_woe"]
].head(20).sort_values("age")

In [ ]:
optb_age.binning_table.build()

In [ ]:
binning_models = {}

In [ ]:
for variable in selected_variables:

    x = train[variable]
    y = train["SeriousDlqin2yrs"]

    optb = OptimalBinning(
        name=variable,
        dtype="numerical"
    )

    optb.fit(x, y)

    binning_models[variable] = optb

In [ ]:
# Woe Datasets

train_woe = pd.DataFrame()

validation_woe = pd.DataFrame()

test_woe = pd.DataFrame()

In [ ]:
for variable in selected_variables:

    train_woe[variable] = (
        binning_models[variable]
        .transform(
            train[variable],
            metric="woe"
        )
    )

In [ ]:
for variable in selected_variables:

    validation_woe[variable] = (
        binning_models[variable]
        .transform(
            validation[variable],
            metric="woe"
        )
    )

In [ ]:
for variable in selected_variables:

    test_woe[variable] = (
        binning_models[variable]
        .transform(
            test[variable],
            metric="woe"
        )
    )

In [ ]:
train_woe["SeriousDlqin2yrs"] = train["SeriousDlqin2yrs"]

validation_woe["SeriousDlqin2yrs"] = validation["SeriousDlqin2yrs"]

test_woe["SeriousDlqin2yrs"] = test["SeriousDlqin2yrs"]

In [ ]:
# checking the woe dataset
train_woe.head()

In [ ]:
# checking for missing values in the woe dataset
train_woe.isna().sum()

In [ ]:
# save the woe datasets
train_woe.to_csv(
    "../data/processed/train_woe.csv",
    index=False
)

validation_woe.to_csv(
    "../data/processed/validation_woe.csv",
    index=False
)

test_woe.to_csv(
    "../data/processed/test_woe.csv",
    index=False
)

## WOE Transformation

All selected predictors were transformed using Weight of Evidence encoding based on the optimal binning structure developed on the training sample.

The same binning definitions were subsequently applied to validation and test datasets to ensure consistency and avoid information leakage.

WOE transformation was selected because:

- It preserves monotonic relationships.
- It provides business interpretability.
- It is the standard transformation used in traditional scorecard development.
- It improves compatibility with logistic regression.